In [ ]:
import os
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
def authenticate_client():
    key = os.getenv("AZURE_TEXT_ANALYTICS_KEY")
    endpoint = os.getenv("AZURE_TEXT_ANALYTICS_ENDPOINT")
    
    credential = AzureKeyCredential(key)
    client = TextAnalyticsClient(endpoint=endpoint, credential=credential)
    return client

text_client = authenticate_client()

documents = ["By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement."]
response = text_client.analyze_sentiment(documents=documents)

print(f"Overall Sentiment Label: {response[0].sentiment}")
print("-" * 30)
print("Confidence Scores: ")
print(f"Positive: {response[0].confidence_scores.positive}")
print(f"Neutral: {response[0].confidence_scores.neutral}")
print(f"Negative: {response[0].confidence_scores.negative}")


Overall Sentiment Label: positive
------------------------------
Confidence Scores: 
Positive: 0.87
Neutral: 0.13
Negative: 0.0


In [1]:
import pandas as pd
import re
import nltk
import numpy as np
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

# nltk.download("stopwords")
# nltk.download("wordnet")

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(cleaned_words)

message = "By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement."
cleaned_message = clean_text(message)

df = pd.read_csv("data.csv")
df['cleaned_text'] = df['Text'].apply(clean_text)

""" TFIDF """
# vectorizer = TfidfVectorizer(max_features=50)
# X = vectorizer.fit_transform(df['cleaned_text']).toarray()
# y = df['Sentiment']
# vectorized_message = vectorizer.transform([cleaned_message]).toarray()


""" Word2Vec"""
tokenized_sentences = [text.split() for text in df['cleaned_text']]
word2vec_model = Word2Vec(sentences=tokenized_sentences, vector_size=50, window=5, min_count=1, workers=4)

def get_sentence_vector(text, model):
    words = text.split()

    word_vectors = [model.wv[word] for word in words if word in model.wv]

    if not word_vectors:
        return np.zeros(model.vector_size)
    
    return np.mean(word_vectors, axis=0)

X = np.array([get_sentence_vector(text, word2vec_model) for text in df['cleaned_text']])
y = df['Sentiment']
vectorized_message = get_sentence_vector(cleaned_message, word2vec_model)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ann_tool = MLPClassifier(hidden_layer_sizes=(100, 50, 20), max_iter=500, random_state=42)
ann_tool.fit(X_train, y_train)

print(f"Model trained! Accuracy: {ann_tool.score(X_test, y_test)}")


cleaned_message = clean_text(message)
predicted_sentiment = ann_tool.predict(vectorized_message.reshape(1, -1))
probability = ann_tool.predict_proba(vectorized_message.reshape(1, -1))
print(f"Predicted Sentiment: {predicted_sentiment[0]}")
print(f"Probability: {probability[0]}")

Model trained! Accuracy: 0.5172413793103449
Predicted Sentiment: negative
Probability: [0.79316664 0.20683339]


In [4]:
import math
import random

def transpune_matrice(A):
    """
    Transpune matricea A, adică transformă rândurile în coloane și invers.
    A: matricea de intrare, reprezentată ca o listă de liste (listă de rânduri).
    Returnează matricea transpusă, tot ca o listă de liste.
    """
    randuri = len(A)
    coloane = len(A[0]) if randuri > 0 else 0
    return [[A[i][j] for i in range(randuri)] for j in range(coloane)]

def inmulteste_matrici(A, B):
    """
    Înmulțește matricele A și B.
    A: prima matrice de intrare, reprezentată ca o listă de liste.
    B: a doua matrice de intrare, reprezentată ca o listă de liste.
    Returnează matricea rezultat, tot ca o listă de liste.
    """
    randuri_A = len(A)
    randuri_B = len(B)
    coloane_A = len(A[0]) if randuri_A > 0 else 0
    coloane_B = len(B[0]) if randuri_B > 0 else 0

    C = [[0.0 for _ in range(coloane_B)] for _ in range(randuri_A)]
    for i in range(randuri_A):
        for j in range(coloane_B):
            for k in range(coloane_A):
                C[i][j] += A[i][k] * B[k][j]
    return C


def sigmoid(x):
    """
    Funcția sigmoidă, care transformă orice valoare reală într-un interval între 0 și 1.
    x: valoarea de intrare (float).
    Returnează valoarea sigmoidă corespunzătoare lui x.
    """
    return 1 / (1 + math.exp(-x))

In [8]:
def initializeaza_ponderi(input_size, hidden_size, output_size):
    """
    Inițializează ponderile și bias-urile pentru rețeaua neuronală.
    input_size: numărul de neuroni din stratul de intrare.
    hidden_size: numărul de neuroni din stratul ascuns.
    output_size: numărul de neuroni din stratul de ieșire.
    Returnează ponderile și bias-urile inițializate.
    """
    limit = math.sqrt(6 / (input_size + hidden_size))
    # limit = 0.1

    W1 = [[random.uniform(-limit, limit) for _ in range(hidden_size)] for _ in range(input_size)]
    b1 = [[0.0 for _ in range(hidden_size)]]

    W2 = [[random.uniform(-limit, limit) for _ in range(output_size)] for _ in range(hidden_size)]
    b2 = [[0.0 for _ in range(output_size)]]

    return W1, b1, W2, b2


def antreneaza_ann_manual(X, Y, hidden_size, epoci, lr):
    """
    Antrenează rețeaua neuronală manual.
    X: matricea de intrare, reprezentată ca o listă de liste.
    Y: matricea de ieșire, reprezentată ca o listă de liste.
    hidden_size: numărul de neuroni din stratul ascuns.
    epoci: numărul de epoci de antrenament.
    lr: rată de învățare.
    Returnează ponderile și bias-urile antrenați.
    """
    input_size = len(X[0])
    output_size = 1
    W1, b1, W2, b2 = initializeaza_ponderi(input_size, hidden_size, output_size)
    num_example = len(X)

    for epoca in range(epoci):

        data_combinate = list(zip(X, Y))
        random.shuffle(data_combinate)

        X_shuffled = [pereche[0] for pereche in data_combinate]
        Y_shuffled = [pereche[1] for pereche in data_combinate]

        Z1 = inmulteste_matrici(X_shuffled, W1)
        A1 = [[sigmoid(Z1[i][j] + b1[0][j]) for j in range(len(Z1[0]))] for i in range(len(Z1))]

        Z2 = inmulteste_matrici(A1, W2)
        A2 = [[sigmoid(Z2[i][j] + b2[0][j]) for j in range(len(Z2[0]))] for i in range(len(Z2))]

        E_iesire = []
        for i in range(len(A2)):
            row = []
            for j in range (len(A2[0])):
                error = (A2[i][j] - Y_shuffled[i][j]) * A2[i][j] * (1 - A2[i][j])
                row.append(error)
            E_iesire.append(row)

        A1_T = transpune_matrice(A1)
        dW2 = inmulteste_matrici(A1_T, E_iesire)

        W2_T = transpune_matrice(W2)
        E_ascunsa_pre = inmulteste_matrici(E_iesire, W2_T)
        E_ascunsa = []
        for i in range(len(E_ascunsa_pre)):
            E_ascunsa.append([E_ascunsa_pre[i][j] * A1[i][j] * (1 - A1[i][j]) for j in range(len(A1[0]))])
        
        X_T = transpune_matrice(X_shuffled)
        dW1 = inmulteste_matrici(X_T, E_ascunsa)

        for i in range(len(W1)):
            for j in range(len(W1[0])):
                W1[i][j] -= (lr * dW1[i][j]) / num_example

        for i in range(len(W2)):
            for j in range(len(W2[0])):
                W2[i][j] -= (lr * dW2[i][j]) / num_example

        for j in range(len(b1[0])):
            eroare_medie_b1 = sum(E_ascunsa[i][j] for i in range(len(E_ascunsa))) / num_example
            b1[0][j] -= lr * eroare_medie_b1

        for j in range(len(b2[0])):
            eroare_medie_b2 = sum(E_iesire[i][j] for i in range(len(E_iesire))) / num_example
            b2[0][j] -= lr * eroare_medie_b2

        if epoca % 10 == 0:
            mse = sum([(A2[i][0] - Y_shuffled[i][0]) ** 2 for i in range(len(Y_shuffled))]) / num_example
            print(f"Epoca {epoca}, Loss(MSE): {mse:.4f}")
    return W1, b1, W2, b2

def impartire_date_manual(X, Y, test_size=0.2):
    """
    Împarte datele într-un set de antrenament și un set de testare.
    X: matricea de intrare.
    Y: matricea de ieșire.
    test_size: proporția de date de testare.
    Returnează cele patru seturi de date.
    """
    date_combinate = list(zip(X, Y))
    random.shuffle(date_combinate)

    prag = int(len(date_combinate) * (1 - test_size))
    date_antrenare = date_combinate[:prag]
    date_testare = date_combinate[prag:]

    x_train = [pereche[0] for pereche in date_antrenare]
    y_train = [[float(pereche[1])] for pereche in date_antrenare]
    x_test = [pereche[0] for pereche in date_testare]
    y_test = [[float(pereche[1])] for pereche in date_testare]

    return x_train, y_train, x_test, y_test

def predictie_manual(X, W1, b1, W2, b2):
    """
    Face o predicție folosind rețeaua neuronală antrenată.
    X: matricea de intrare, reprezentată ca o listă de liste.
    W1, b1, W2, b2: ponderile și bias-urile rețelei neuronale.
    Returnează predicțiile pentru fiecare exemplu din X.
    """
    Z1 = inmulteste_matrici(X, W1)
    A1 = [[sigmoid(Z1[i][j] + b1[0][j]) for j in range(len(Z1[0]))] for i in range(len(Z1))]

    Z2 = inmulteste_matrici(A1, W2)
    A2 = [[sigmoid(Z2[i][j] + b2[0][j]) for j in range(len(Z2[0]))] for i in range(len(Z2))]

    return [1 if A2[i][0] >= 0.5 else 0 for i in range(len(A2))]

def afiseaza_probabilitati(X, W1, b1, W2, b2):
    """
    Afișează probabilitățile pentru fiecare exemplu din X folosind rețeaua neuronală antrenată.
    X: matricea de intrare, reprezentată ca o listă de liste.
    W1, b1, W2, b2: ponderile și bias-urile rețelei neuronale.
    """
    Z1 = inmulteste_matrici(X, W1)
    A1 = [[sigmoid(Z1[i][j] + b1[0][j]) for j in range(len(Z1[0]))] for i in range(len(Z1))]

    Z2 = inmulteste_matrici(A1, W2)
    A2 = [[sigmoid(Z2[i][j] + b2[0][j]) for j in range(len(Z2[0]))] for i in range(len(Z2))]

    for i in range(len(A2)):
        print(f"Exemplul {i + 1}: Probabilitate pozitivă = {A2[i][0]:.4f}, Probabilitate negativă = {1 - A2[i][0]:.4f}")


In [11]:
y_manual = [1.0 if sentiment == 'positive' else 0.0 for sentiment in df['Sentiment']]

X_manual = X.tolist()

X_train_m, y_train_m, X_test_m, y_test_m = impartire_date_manual(X_manual, y_manual)

W1, b1, W2, b2 = antreneaza_ann_manual(X_train_m, y_train_m, hidden_size=100, epoci=3000, lr=1.0)

target_X = vectorized_message.tolist()

prediction = predictie_manual(target_X, W1, b1, W2, b2)

sentiment = "positive" if prediction[0] == 1 else "negative"
print(f"Manual ANN Result: {sentiment}")
afiseaza_probabilitati(target_X, W1, b1, W2, b2)

Epoca 0, Loss(MSE): 0.2653
Epoca 10, Loss(MSE): 0.2498
Epoca 20, Loss(MSE): 0.2494
Epoca 30, Loss(MSE): 0.2490
Epoca 40, Loss(MSE): 0.2486
Epoca 50, Loss(MSE): 0.2481
Epoca 60, Loss(MSE): 0.2477
Epoca 70, Loss(MSE): 0.2473
Epoca 80, Loss(MSE): 0.2469
Epoca 90, Loss(MSE): 0.2464
Epoca 100, Loss(MSE): 0.2460
Epoca 110, Loss(MSE): 0.2456
Epoca 120, Loss(MSE): 0.2451
Epoca 130, Loss(MSE): 0.2446
Epoca 140, Loss(MSE): 0.2442
Epoca 150, Loss(MSE): 0.2437
Epoca 160, Loss(MSE): 0.2432
Epoca 170, Loss(MSE): 0.2426
Epoca 180, Loss(MSE): 0.2421
Epoca 190, Loss(MSE): 0.2415
Epoca 200, Loss(MSE): 0.2410
Epoca 210, Loss(MSE): 0.2404
Epoca 220, Loss(MSE): 0.2398
Epoca 230, Loss(MSE): 0.2392
Epoca 240, Loss(MSE): 0.2385
Epoca 250, Loss(MSE): 0.2378
Epoca 260, Loss(MSE): 0.2371
Epoca 270, Loss(MSE): 0.2364
Epoca 280, Loss(MSE): 0.2357
Epoca 290, Loss(MSE): 0.2349
Epoca 300, Loss(MSE): 0.2341
Epoca 310, Loss(MSE): 0.2333
Epoca 320, Loss(MSE): 0.2325
Epoca 330, Loss(MSE): 0.2316
Epoca 340, Loss(MSE): 0.2